In [ ]:
import numpy as np
import tifffile
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LightSource

In [2]:
with tifffile.TiffFile('fuji.tif') as tif:
    elevation = tif.pages[0].asarray().astype(np.float32)
    tp = tif.pages[0].tags['ModelTiepointTag'].value
    sc = tif.pages[0].tags['ModelPixelScaleTag'].value

nrows, ncols = elevation.shape
lon = tp[3] + np.arange(ncols) * sc[0]
lat = tp[4] - np.arange(nrows) * sc[1]

print(f"Сетка: {nrows}x{ncols}")
print(f"Широта: {lat[-1]:.4f}° — {lat[0]:.4f}°N")
print(f"Долгота: {lon[0]:.4f}° — {lon[-1]:.4f}°E")
print(f"Высота: {elevation.min():.0f} — {elevation.max():.0f} м")

lat0, lon0 = 35.3606, 138.7274
km_per_deg_lat = 111.0
km_per_deg_lon = 111.0 * np.cos(np.radians(lat0))

LON, LAT = np.meshgrid(lon, lat)
X_km = (LON - lon0) * km_per_deg_lon
Y_km = (LAT - lat0) * km_per_deg_lat

np.save("data/fuji_elevation.npy", elevation)
np.save("data/fuji_lat.npy", lat)
np.save("data/fuji_lon.npy", lon)
print("Сохранено: fuji_elevation.npy, fuji_lat.npy, fuji_lon.npy")

Сетка: 1080x1080
Широта: 35.2004° — 35.5001°N
Долгота: 138.5999° — 138.8996°E
Высота: 63 — 3759 м
Сохранено: fuji_elevation.npy, fuji_lat.npy, fuji_lon.npy


In [7]:
cmap_terrain = LinearSegmentedColormap.from_list('fuji', [
    (0.00, '#1a472a'),  # тёмно-зелёный (лес)
    (0.08, '#2d6a3e'),  # зелёный
    (0.15, '#4a8c3f'),  # светло-зелёный
    (0.25, '#7a9a4a'),  # жёлто-зелёный
    (0.35, '#a08050'),  # охра
    (0.50, '#8b6942'),  # коричневый
    (0.65, '#a0896a'),  # светлый камень
    (0.80, '#b8a898'),  # серо-бежевый
    (0.90, '#d0d0d0'),  # серый (скалы)
    (1.00, '#ffffff'),  # белый (снег)
])

# Прореживаем: 1080 -> ~270 (каждый 4-й)
step = 4
Zs = elevation[::step, ::step]
Xs = X_km[::step, ::step]
Ys = Y_km[::step, ::step]

fig = plt.figure(figsize=(18, 11))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(
    Xs, Ys, Zs,
    cmap=cmap_terrain,
    rstride=1, cstride=1,
    linewidth=0,
    antialiased=True,
    alpha=0.95,
    vmin=0, vmax=3800,
)

ax.set_xlabel('<- Запад     Восток -> (км)', fontsize=11, labelpad=12)
ax.set_ylabel('<- Юг     Север -> (км)', fontsize=11, labelpad=12)
ax.set_zlabel('Высота (м)', fontsize=11, labelpad=10)
ax.set_title('Гора Фудзи — 3D-модель по данным SRTM\n'
             f'Сетка {nrows}x{ncols}, разрешение ~30 м',
             fontsize=14, fontweight='bold')

ax.view_init(elev=30, azim=225)
ax.set_box_aspect([1, 1, 0.45])

fig.colorbar(surf, ax=ax, shrink=0.45, aspect=15,
             label='Высота (м)', pad=0.08)

plt.tight_layout()
plt.savefig("assets/fuji_3d_real.png", dpi=200, bbox_inches='tight')
plt.close()
print("assets/fuji_3d_real.png")

assets/fuji_3d_real.png


In [12]:
fig3, ax3 = plt.subplots(figsize=(11, 9))

ls = LightSource(azdeg=315, altdeg=35)
rgb = ls.shade(elevation, cmap=cmap_terrain, blend_mode='overlay',
               vmin=0, vmax=3800, vert_exag=2)

ax3.imshow(rgb, extent=[lon[0], lon[-1], lat[-1], lat[0]], aspect='auto')

# Изолинии
cs = ax3.contour(LON, LAT, elevation,
                 levels=np.arange(500, 4000, 500),
                 colors='k', linewidths=0.5, alpha=0.4)
ax3.clabel(cs, fmt='%d м', fontsize=8)

ax3.plot(138.7274, 35.3606, 'r^', ms=14, zorder=5,
         markeredgecolor='black', markeredgewidth=1)
ax3.annotate('  Вершина\n  3776 м', xy=(138.7274, 35.3606),
             fontsize=10, fontweight='bold', color='darkred')

ax3.set_xlabel('Долгота (°E)', fontsize=12)
ax3.set_ylabel('Широта (°N)', fontsize=12)
ax3.set_title('Гора Фудзи — карта высот с подсветкой рельефа',
              fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig("assets/fuji_heatmap_real.png", dpi=200, bbox_inches='tight')
plt.close()
print("assets/fuji_heatmap_real.png")

C:\Users\joinm\AppData\Local\Temp\ipykernel_1440\885994062.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
